In [1]:
import re
import pandas as pd 
from tqdm import tqdm 

## 1. Finding the Unique Words 

In [2]:
with open('big.txt','r') as fd:
    lines = fd.readlines()
    words =[]
    for line in lines  :
        words += re.findall('\w+',line.lower())
print(len(words))
vocab = list(set(words))
print(len(vocab))

<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:5: SyntaxWarning: invalid escape sequence '\w'
C:\Users\v\AppData\Local\Temp\ipykernel_15496\3919749970.py:5: SyntaxWarning: invalid escape sequence '\w'
  words += re.findall('\w+',line.lower())


1115585
32198


## 2. Finding the Probability Distribution 


In [3]:
word_probability = {}

for word in tqdm(vocab):
    word_probability[word] = float(words.count(word)/len(words))

100%|████████████████████████████████████████████████████████████████████████████| 32198/32198 [12:43<00:00, 42.16it/s]


## 3. Text Preprocessing 

# Splitting 

In [5]:
def split(word):
    parts =[]
    for i in range(len(word)+1):
        parts += [(word[ : i],word[i: ])]
    return parts     

## 3.1 Delete 

'loave' -> 'love'

In [6]:
def delete(word):
    output =[]
    for l,r in split(word):
        output.append(l+r[1:])
    return output
delete('loave')    

['oave', 'lave', 'love', 'loae', 'loav', 'loave']

## 3.2 Swap
'lvoe' -> 'love'

In [7]:
def swap(word):
    output =[]
    for l,r in split(word):
        if (len(r) > 1):
            output.append(l+r[1]+r[0]+r[2:])
    return output
swap('lvoe')    

['vloe', 'love', 'lveo']

## 3.3 Replace

'lave' -> 'love'

In [8]:
def replace(word):
    characters = 'abcdefghijklmnopqrstuvwxyz'
    output =[]
    for l,r in split(word):
        for char in characters:
            output.append(l+ char + r[1:])
    return output 
len(replace('lave'))    

130

## 3.4 Insert 
'lve' -> 'love'

In [9]:
def insert(word):
    characters = 'abcdefghijklmnopqrstuvwxyz'
    output =[]
    for l,r in split(word):
        for char in characters:
            output.append(l+ char + r)
    return output 
len(insert('lve'))    

104

## 4. Finding the Prediction(Level - 1)


## 4.1) Combining Possible Words 

In [10]:
def edit(word):
    return list(set(insert(word)+delete(word) + swap(word) + replace(word)))

## 4.2) Predicting the Word 

In [15]:
def spell_check_edit_1(word,count =5):
    output =[]
    suggested_words = edit(word)

    for wrd in suggested_words:
        if wrd in word_probability.keys():
            output.append([wrd,word_probability[wrd]])
    return list(pd.DataFrame(output,columns=['word','prob']).sort_values(by ='prob' , ascending =False).head(count)['word'].values)         

In [16]:
spell_check_edit_1('famili')

['family']

In [18]:
spell_check_edit_1('geve')

['give', 'gave', 'eve', 'neve', 'gee']

## 5 Finding the Prediction (Level - 2 )

## 5.1) Combining Possible Words

In [19]:
def spell_check_edit_2 (word, count=5):
    output =[]
    suggested_words = edit(word) # level one  edit 
    for e1 in edit(word):
        suggested_words += edit(e1) # second level Edit 
    suggested_word = list(set(suggested_words))

    for wrd in suggested_word:
        if wrd in word_probability.keys():
            output.append([wrd,word_probability[wrd]])
    return list(pd.DataFrame(output,columns = ['word','prob']).sort_values(by='prob',ascending =False).head(count)['word'].values
               )
spell_check_edit_2('fameli')    

['family', 'namely', 'fame', 'camel', 'amelie']

In [20]:
spell_check_edit_2('honet')

['one', 'bone', 'done', 'money', 'home']